In [1]:
!git clone https://github.com/yohanshin/WHAM.git
%cd WHAM

fatal: destination path 'WHAM' already exists and is not an empty directory.
/content/WHAM


In [2]:
# 1. Tạo thư mục
!mkdir -p checkpoints
!mkdir -p dataset/body_models

# 2. Tải file WHAM Checkpoint (Cái này là quan trọng nhất)
!pip install gdown
!gdown --id 19qkI-a6xuwob9_RFNSPWf1yWErwVVlks -O checkpoints/wham_vit_bedlam_w_3dpw.pth.tar

# 3. Tải Auxiliary data (Mấy cái râu ria để chạy SMPL)
!gdown --id 1pbmzRbWGgae6noDIyQOnohzaVnX_csUZ -O dataset/body_models.tar.gz
!tar -xvf dataset/body_models.tar.gz -C dataset/

/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=19qkI-a6xuwob9_RFNSPWf1yWErwVVlks
From (redirected): https://drive.google.com/uc?id=19qkI-a6xuwob9_RFNSPWf1yWErwVVlks&confirm=t&uuid=d960a1a9-d9d1-4bf5-a48b-38190748ac6e
To: /content/WHAM/checkpoints/wham_vit_bedlam_w_3dpw.pth.tar
100% 527M/527M [00:04<00:00, 130MB/s]
/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From: https://drive.google.com/uc?id=1pbmzRbWGgae6noDIyQOnohzaVnX_csUZ
To: /content/WHAM/dataset/body_models.tar.gz
100% 964k/964k [00:00<00:00, 10.1MB/s]
body_models/
body_models/coco_aug_dict.pth
body_models/J

In [3]:
# 1. Tạo đúng cái "chuồng" mà model nó đòi
!mkdir -p /content/WHAM/dataset/body_models/smpl

# 2. Tao sẽ dùng một cái mẹo: Nếu mày chưa có file SMPL_NEUTRAL.pkl xịn,
# tao sẽ chỉ mày cách lấy nó từ một nguồn public khác (vì mày đang khô máu mà)
!wget https://github.com/vchoutas/smplx/raw/master/models/smpl/SMPL_NEUTRAL.pkl -P /content/WHAM/dataset/body_models/smpl/

# 3. Kiểm tra lại lần nữa
print("--- Check lại chuồng xem có 'xương' chưa ---")
!ls -R /content/WHAM/dataset/body_models/smpl/

--2026-03-23 12:39:30--  https://github.com/vchoutas/smplx/raw/master/models/smpl/SMPL_NEUTRAL.pkl
Resolving github.com (github.com)... 140.82.114.3
Connecting to github.com (github.com)|140.82.114.3|:443... connected.
HTTP request sent, awaiting response... 404 Not Found
2026-03-23 12:39:30 ERROR 404: Not Found.

--- Check lại chuồng xem có 'xương' chưa ---
/content/WHAM/dataset/body_models/smpl/:
SMPL_NEUTRAL.pkl


In [4]:
!pip install coremltools chumpy loguru smplx -q

In [5]:
!pip install yacs

In [6]:
import inspect
import torch
import torch.serialization
import numpy as np
import coremltools as ct
from coremltools.converters.mil import register_torch_op
from coremltools.converters.mil.mil import Builder as mb
import types
import warnings

warnings.filterwarnings("ignore")

# ==========================================
# 1. TẨY RỬA MÔI TRƯỜNG & ĐỒ CỔ (Giữ nguyên)
# ==========================================
np.bool, np.int, np.float = bool, int, float
np.complex, np.object, np.unicode, np.str = complex, object, str, str
if not hasattr(inspect, 'getargspec'): inspect.getargspec = inspect.getfullargspec
if hasattr(torch.serialization, 'load'): torch.load = torch.serialization.load
torch.serialization.add_safe_globals([np.core.multiarray.scalar, np.dtype, np.ndarray])
_real_load = torch.serialization.load
def _safe_load(*args, **kwargs):
    kwargs['weights_only'] = False
    return _real_load(*args, **kwargs)
torch.load = _safe_load

# ==========================================
# 2. VÁ LỖI TOÁN TỬ (.mT và view_as)
# ==========================================
@register_torch_op(override=True)
def mt(context, node):
    x = context[node.inputs[0]]
    perm = list(range(len(x.shape)))
    perm[-1], perm[-2] = perm[-2], perm[-1]
    return mb.transpose(x=x, perm=perm, name=node.name)

@register_torch_op(override=True)
def view_as(context, node):
    x, y_ref = context[node.inputs[0]], context[node.inputs[1]]
    return mb.reshape(x=x, shape=mb.shape(x=y_ref), name=node.name)

# ==========================================
# 3. KHỞI TẠO MÔ HÌNH (VỚI VŨ KHÍ MỚI)
# ==========================================
from configs.config import get_cfg_defaults
from lib.models import build_network, build_body_model

print("1. Đang khởi tạo mô hình...")
cfg = get_cfg_defaults()
cfg.merge_from_file('configs/yamls/demo.yaml')
smpl = build_body_model(cfg.DEVICE, 1)
network = build_network(cfg, smpl)

# CHIẾN THUẬT "VƯỜN KHÔNG NHÀ TRỐNG":
# Vô hiệu hóa mọi thứ trong LSTM có thể khiến Apple "ngứa tay" tối ưu lỗi
for name, module in network.named_modules():
    if isinstance(module, torch.nn.LSTM):
        print(f"--- Đang bảo vệ lớp LSTM: {name} ---")
        module.dropout = 0
        # Ép buộc tính toán lại flat_weights để tránh lỗi bộ nhớ
        module.flatten_parameters()

network.eval()

# Vá lỗi masking
def patched_preprocess(self, x, mask):
    self.b, self.f = x.shape[:2]
    _mask = mask.bool().unsqueeze(-1).repeat(1, 1, 1, 2).reshape(self.b, self.f, -1)
    _mask = torch.cat((_mask, torch.zeros_like(_mask[..., :3])), dim=-1)
    _mask_emb = (mask.bool().unsqueeze(-1) * self.mask_embedding).reshape(self.b, self.f, -1)
    _mask_emb = torch.cat((_mask_emb, torch.zeros_like(_mask_emb[..., :3])), dim=-1)
    return torch.where(_mask, torch.zeros_like(x), x) + _mask_emb

network.preprocess = types.MethodType(patched_preprocess, network)

# ==========================================
# 4. TRACING & CONVERT (BẢN TRẢ THÙ: HACK RAM TẬN GỐC)
# ==========================================
import gc
import sys

class CoreMLWrapper(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
    def forward(self, x, init_kp, init_smpl, img_features, mask, init_root, cam_angvel, cam_intrinsics, bbox, res):
        out = self.model(x, (init_kp, init_smpl), img_features, mask, init_root, cam_angvel, cam_intrinsics=cam_intrinsics, bbox=bbox, res=res, refine_traj=False)
        return (out['pose'], out['betas'], out['poses_root_world'], out['trans_world'])

print("1. Đang đóng băng đồ thị (Tracing)...")
dummy_args = (
    torch.randn(1, 81, 37), torch.randn(1, 1, 88), torch.randn(1, 1, 24, 6),
    torch.randn(1, 81, 1024), torch.ones(1, 81, 17, dtype=torch.bool),
    torch.randn(1, 1, 6), torch.randn(1, 81, 6),
    torch.eye(3).reshape(1, 1, 3, 3), torch.randn(1, 81, 3), torch.tensor([[1080.0, 1920.0]])
)

wrapped_network = CoreMLWrapper(network).cpu()
traced_model = torch.jit.trace(wrapped_network, dummy_args, strict=False)

print("2. Đang dọn rác RAM...")
del network
del wrapped_network
gc.collect()

# --- ĐẠI PHÁP HACK RAM: DIỆT BUG APPLE TỪ BỘ NHỚ ---
print("3. Đang chui vào RAM để tẩy trắng fuse_stack_split...")

# Ép CoreML load hết các module ngầm vào RAM
try: import coremltools.converters.mil.mil.passes.defs.common
except: pass

patched = False
# Quét toàn bộ Object đang tồn tại trong RAM của Colab
for obj in gc.get_objects():
    if isinstance(obj, type): # Tìm các Class
        name = getattr(obj, '__name__', '').lower()
        if 'fuse_stack_split' in name:
            print(f"🔥 ĐÃ TÚM CỔ ĐƯỢC KẺ THÙ: {obj.__name__}")
            # Tẩy trắng toàn bộ não của nó
            setattr(obj, '__call__', lambda self, *args, **kwargs: None)
            setattr(obj, 'apply', lambda self, *args, **kwargs: None)
            patched = True

if not patched:
    print("⚠️ Cảnh báo: Không tìm thấy bug, cầu nguyện Apple đã tự fix!")

print("4. Bắt đầu Convert (Để mặc định cho Xcode tạo Schema, ép FP16 chống OOM)...")
input_specs = [ct.TensorType(name=f"in_{i}", shape=arg.shape) for i, arg in enumerate(dummy_args)]

# Ép luôn tên Output xịn xò vào Schema để Xcode sướng mắt
output_specs = [
    ct.TensorType(name="pose_6d"),
    ct.TensorType(name="shape_betas"),
    ct.TensorType(name="world_rotation"),
    ct.TensorType(name="world_translation")
]

# LƯU Ý: KHÔNG DÙNG EMPTY NỮA. Để nó chạy pipeline sinh Metadata!
mlmodel = ct.convert(
    traced_model,
    inputs=input_specs,
    outputs=output_specs,
    convert_to="mlprogram",
    compute_precision=ct.precision.FLOAT16
)

print("5. Dọn dẹp Metadata lần cuối...")
mlmodel.author = "Long"
mlmodel.short_description = "WHAM 3D Human Pose - Vượt Kiếp Nạn"

print("6. Lưu file...")
mlmodel.save("WHAM_FINAL_VICTORY.mlpackage")
print("🎉 ĐẠI CÔNG CÁO THÀNH! Tải ngay và ném vào Xcode đi ông!!!")

1. Đang khởi tạo mô hình...


2026-03-23 12:40:11.491 | INFO     | lib.models:build_network:36 - => loaded checkpoint 'checkpoints/wham_vit_bedlam_w_3dpw.pth.tar' 
Running MIL backend_mlprogram pipeline: 100%|██████████| 12/12 [00:03<00:00,  3.49 passes/s]


In [8]:
import os

# 1. Nén cái folder thành file .zip
os.system("zip -r WHAM_Final_iOS.zip WHAM_FINAL_VICTORY.mlpackage")

# 2. Kích hoạt lệnh tải xuống của Colab
from google.colab import files
files.download("WHAM_Final_iOS.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>